<a href="https://colab.research.google.com/github/cstecker/politicsRLab/blob/main/03%20-%20Parteien%2C%20Wahlen%20und%20Regierungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Parteien, Wahlen, Parteiensysteme und Cleavages

Dieses Notebook begleitet die Sitzung zu **Parteien und Konfliktlinien**. Es übersetzt zentrale Konzepte aus der Vorlesung in kleine, nachvollziehbare R-Beispiele.

Wir arbeiten mit vier Leitfragen:

1. Wie kann man Parteiensysteme anhand von Zahl und Größe der Parteien beschreiben?
2. Was misst die effektive Parteienzahl?
3. Wie misst man Disproportionalität zwischen Stimmen und Sitzen?
4. Wie lassen sich Parteipositionen und Cleavages im zweidimensionalen Raum untersuchen?

Der wichtigste Datensatz ist **PPEG 2025v1**. Für Parteipositionen nutzen wir zusätzlich eine lokale ParlGov-CHES-Verknüpfung, wenn sie verfügbar ist.


## 1. Vorbereitung

Die erste Codezelle lädt die benötigten Pakete. Wenn ein Paket noch fehlt, wird es installiert. Das kann in Colab beim ersten Durchlauf kurz dauern.


In [ ]:
cran_packages <- c(
  "tibble",
  "dplyr",
  "tidyr",
  "purrr",
  "stringr",
  "forcats",
  "readr",
  "ggplot2",
  "ggrepel",
  "scales"
)

missing_packages <- cran_packages[
  !vapply(cran_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_packages) > 0) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}

# ggparliament ist nicht für jede R-Version auf CRAN verfügbar.
# Falls CRAN es nicht liefert, versuchen wir die GitHub-Version.
if (!requireNamespace("ggparliament", quietly = TRUE)) {
  if (!requireNamespace("remotes", quietly = TRUE)) {
    install.packages("remotes", repos = "https://cloud.r-project.org")
  }

  try(
    remotes::install_github("robwhickman/ggparliament", upgrade = "never"),
    silent = TRUE
  )
}

invisible(lapply(cran_packages, library, character.only = TRUE))

has_ggparliament <- requireNamespace("ggparliament", quietly = TRUE)

if (has_ggparliament) {
  library(ggparliament)
} else {
  message("ggparliament ist nicht verfügbar. Das Notebook nutzt einfache Punktdiagramme als Fallback.")
}


## 2. Zwei kleine Funktionen: ENP und Gallagher-Index

Die Formeln sind nicht lang. Deshalb schreiben wir sie zuerst selbst. So ist klar, was R später in den größeren Beispielen macht.

Die **effektive Parteienzahl** nach Laakso/Taagepera lautet:

\[
ENP = \frac{1}{\sum_i p_i^2}
\]

`p_i` ist der Anteil einer Partei. Die Anteile können als Prozentwerte eingegeben werden; die Funktion normiert sie intern auf Proportionen.


In [ ]:
effective_number <- function(shares) {
  shares <- shares[!is.na(shares) & shares > 0]
  proportions <- shares / sum(shares)

  1 / sum(proportions^2)
}

effective_number(c(25, 25, 25, 25))


Der **Gallagher-Index** misst, wie stark Stimmen- und Sitzanteile voneinander abweichen:

\[
G = \sqrt{\frac{1}{2}\sum_i (v_i - s_i)^2}
\]

`v_i` ist der Stimmenanteil, `s_i` der Sitzanteil einer Partei. In dieser Übungsfunktion müssen beide Summen 100 ergeben. Das ist Absicht: Die Fehlermeldung hilft, die Logik des Index zu verstehen.


In [ ]:
check_sum_100 <- function(x, name) {
  total <- sum(x)

  if (abs(total - 100) > 0.001) {
    stop(name, " muss 100 ergeben. Aktuelle Summe: ", round(total, 2))
  }
}

gallagher_index <- function(votes, seats) {
  check_sum_100(votes, "votes")
  check_sum_100(seats, "seats")

  sqrt(0.5 * sum((votes - seats)^2))
}

gallagher_index(
  votes = c(50, 50),
  seats = c(50, 50)
)


## 3. Experiment: Zahl und Größe von Parteien verändern

Ändern Sie in der folgenden Zelle nur `party_names` und `party_shares`. Danach die Zelle neu ausführen.

Wichtig: Die Werte in `party_shares` müssen nicht zwingend 100 ergeben. Die ENP-Funktion normiert sie automatisch. Für die Interpretation ist es aber meist einfacher, mit Prozentwerten zu arbeiten.


In [ ]:
party_names <- c("A", "B", "C", "D")
party_shares <- c(25, 25, 25, 25)

party_system <- tibble(
  party = party_names,
  share = party_shares
)

party_system


In [ ]:
enp_value <- effective_number(party_system$share)

enp_value


In [ ]:
ggplot(party_system, aes(x = party, y = share)) +
  geom_col(fill = "#2c7fb8", width = 0.7) +
  labs(
    title = paste("Effektive Parteienzahl:", round(enp_value, 2)),
    x = "Partei",
    y = "Relative Größe"
  ) +
  theme_minimal(base_size = 12)


### Aufgabe 1: ENP verstehen

Arbeiten Sie in Gruppen mit der vorherigen Zelle.

1. Erzeugen Sie ein Zweiparteiensystem mit zwei gleich großen Parteien. Welche ENP entsteht?
2. Erzeugen Sie ein Vierparteiensystem mit vier gleich großen Parteien. Welche ENP entsteht?
3. Erzeugen Sie ein Vierparteiensystem mit den Größen `45, 45, 5, 5`. Warum ist die ENP deutlich kleiner als 4?
4. Erzeugen Sie ein System mit sieben Parteien, aber einer dominanten Partei. Ab wann fühlt sich das System eher wie ein dominantes Parteiensystem an?


## 4. Experiment: Disproportionalität selbst ausprobieren

Hier können Sie Stimmen- und Sitzanteile selbst verändern. Anders als bei der ENP müssen `votes` und `seats` jeweils exakt 100 ergeben.

Interpretation: Je höher der Gallagher-Index, desto stärker weicht die Sitzverteilung von der Stimmenverteilung ab.


In [ ]:
parties <- c("A", "B", "C", "D")
votes <- c(40, 30, 20, 10)
seats <- c(55, 35, 10, 0)

disprop_example <- tibble(
  party = parties,
  votes = votes,
  seats = seats,
  difference = seats - votes
)

disprop_example


In [ ]:
gallagher_value <- gallagher_index(
  votes = disprop_example$votes,
  seats = disprop_example$seats
)

gallagher_value


In [ ]:
disprop_long <- disprop_example |>
  select(party, votes, seats) |>
  pivot_longer(
    cols = c(votes, seats),
    names_to = "measure",
    values_to = "percent"
  ) |>
  mutate(
    measure = recode(measure, "votes" = "Stimmen", "seats" = "Sitze")
  )

ggplot(disprop_long, aes(x = party, y = percent, fill = measure)) +
  geom_col(position = "dodge", width = 0.7) +
  scale_y_continuous(labels = scales::label_percent(scale = 1)) +
  scale_fill_manual(values = c("Stimmen" = "grey70", "Sitze" = "#2c7fb8")) +
  labs(
    title = paste("Gallagher-Index:", round(gallagher_value, 2)),
    x = "Partei",
    y = "Anteil",
    fill = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "top")


### Aufgabe 2: Gallagher-Index verstehen

1. Setzen Sie `votes` und `seats` gleich. Was passiert mit dem Gallagher-Index?
2. Geben Sie Partei A eine deutliche Sitzprämie. Was passiert?
3. Lassen Sie eine Partei 8 Prozent der Stimmen, aber 0 Sitze bekommen. Was passiert?
4. Diskutieren Sie: Warum kann ein Wahlsystem Parteienzahl und Regierungsbildung stark beeinflussen, obwohl sich die Stimmenanteile gar nicht Ändern?


## 5. PPEG-Daten laden

Nun wechseln wir von selbst erfundenen Beispielen zu echten Wahldaten. Das Notebook sucht zuerst lokal nach PPEG-Dateien. Wenn keine lokale Datei gefunden wird, lädt es die offiziellen PPEG-2025v1-Parlamentsdaten herunter.


In [ ]:
first_existing <- function(paths) {
  existing_paths <- paths[file.exists(paths)]

  if (length(existing_paths) == 0) {
    return(NA_character_)
  }

  normalizePath(existing_paths[1], winslash = "/", mustWork = TRUE)
}

load_ppeg_parliament <- function() {
  local_path <- first_existing(c(
    "data/ppeg_parl_2025v1.rds",
    "data/ppeg_elect_cabinet_party.rds",
    "D:/PolScience/Projekte/DATEN/parlgov2/ppeg/raw/ppeg_parl_2025v1/ppeg_parl_2025v1.rds",
    "D:/PolScience/Projekte/DATEN/parlgov2/ppeg/output/ppeg_elect_cabinet_party.rds"
  ))

  if (!is.na(local_path)) {
    message("Nutze lokale PPEG-Datei: ", local_path)
    data <- readRDS(local_path)
  } else {
    message("Keine lokale PPEG-Datei gefunden. Lade PPEG 2025v1 herunter.")

    zip_path <- file.path(tempdir(), "ppeg_parl_2025v1.zip")
    out_dir <- file.path(tempdir(), "ppeg_parl_2025v1")

    download.file(
      url = "https://ppeg.wzb.eu/www/ppeg_parl_2025v1.zip",
      destfile = zip_path,
      mode = "wb"
    )

    unzip(zip_path, exdir = tempdir())

    data <- readr::read_csv(
      file.path(out_dir, "ppeg_parl_2025v1.csv"),
      locale = readr::locale(encoding = "UTF-8"),
      show_col_types = FALSE
    )
  }

  data
}

ppeg_raw <- load_ppeg_parliament()


Die nächste Zelle bringt die wichtigsten Variablen in eine einheitliche Form. Der kombinierte lokale Datensatz enthält manche Parteien mehrfach, weil eine Wahl mit mehreren späteren Kabinetten verknüpft sein kann. Deshalb behalten wir pro Land, Wahl und Partei nur eine Zeile.


In [ ]:
ppeg_parl <- ppeg_raw |>
  mutate(
    election_date = if ("election_date" %in% names(ppeg_raw)) election_date else as.Date(edate),
    country_name = if ("country_name" %in% names(ppeg_raw)) country_name else cname_en,
    party_name_short = if ("party_name_short" %in% names(ppeg_raw)) party_name_short else pinitials,
    party_name_english = if ("party_name_english" %in% names(ppeg_raw)) party_name_english else pname_en,
    vote_share = if ("vote_share" %in% names(ppeg_raw)) vote_share else v_share_wgt * 100,
    seat_share = if ("seat_share" %in% names(ppeg_raw)) seat_share else s_share * 100,
    election_year = as.integer(format(election_date, "%Y")),
    party_label = coalesce(
      na_if(as.character(party_name_short), ""),
      na_if(as.character(party_name_english), ""),
      as.character(party_id)
    )
  ) |>
  filter(!is.na(election_date), !is.na(party_id)) |>
  distinct(iso3c, election_date, party_id, .keep_all = TRUE)

ppeg_parl |>
  select(iso3c, country_name, election_date, party_label, vote_share, seat_share, seats, total_seats) |>
  head(10)


In [ ]:
available_countries <- ppeg_parl |>
  distinct(iso3c, country_name) |>
  arrange(country_name)

available_countries |>
  filter(iso3c %in% c("DEU", "GBR", "USA", "ISR", "NLD"))


## 6. Sitzverteilungen: UK, USA, Deutschland, Israel

Die Folien unterscheiden Zweiparteiensysteme und Mehrparteiensysteme. Wir betrachten hier vier Beispiele:

- UK und USA als stark majoritäre Systeme.
- Deutschland und Israel als Mehrparteiensysteme.

Die Funktion `get_latest_election()` holt für ein Land die jüngste im Datensatz verfügbare Wahl. Für die Sitzgrafik nutzt das Notebook `ggparliament`, falls das Paket verfügbar ist; sonst wird ein einfaches Punktdiagramm mit einem Punkt pro Sitz gezeichnet.


In [ ]:
get_latest_election <- function(data, country_code) {
  one_country <- data |>
    filter(
      iso3c == country_code,
      !is.na(seats),
      seats > 0,
      !is.na(total_seats),
      total_seats > 0
    )

  latest_date <- max(one_country$election_date, na.rm = TRUE)

  one_country |>
    filter(election_date == latest_date) |>
    arrange(desc(seats))
}

get_latest_election(ppeg_parl, "DEU") |>
  select(country_name, election_date, party_label, seats, total_seats, vote_share, seat_share)


In [ ]:
example_countries <- c("GBR", "USA", "DEU", "ISR")

seat_examples <- purrr::map_dfr(
  example_countries,
  ~ get_latest_election(ppeg_parl, .x)
) |>
  mutate(
    country_label = case_when(
      iso3c == "GBR" ~ "Vereinigtes Königreich",
      iso3c == "USA" ~ "USA",
      iso3c == "DEU" ~ "Deutschland",
      iso3c == "ISR" ~ "Israel",
      TRUE ~ country_name
    ),
    panel_label = paste0(country_label, " (", election_year, ")"),
    seat_share = seats / total_seats * 100
  )

seat_examples |>
  select(panel_label, party_label, seats, total_seats, vote_share, seat_share)


In [ ]:
make_parliament_data <- function(data) {
  data <- data |>
    arrange(desc(seats))

  if (has_ggparliament) {
    parliament_data <- ggparliament::parliament_data(
      election_data = data,
      type = "semicircle",
      party_seats = data$seats,
      parl_rows = 8
    )

    parliament_data$panel_label <- data$panel_label[1]

    return(parliament_data)
  }

  # Fallback: ein Punkt pro Sitz in einem einfachen Sitzraster.
  data |>
    mutate(seats = as.integer(seats)) |>
    tidyr::uncount(weights = seats, .id = "seat_in_party") |>
    mutate(
      seat_id = row_number(),
      x = (seat_id - 1) %% 35,
      y = -floor((seat_id - 1) / 35)
    )
}

parliament_plot_data <- seat_examples |>
  group_split(panel_label) |>
  purrr::map_dfr(make_parliament_data)

if (has_ggparliament) {
  seat_plot <- ggplot(parliament_plot_data, aes(x = x, y = y, colour = party_label)) +
    ggparliament::geom_parliament_seats(size = 2.1) +
    coord_equal() +
    theme_void(base_size = 12)
} else {
  seat_plot <- ggplot(parliament_plot_data, aes(x = x, y = y, colour = party_label)) +
    geom_point(size = 1.4) +
    coord_equal() +
    theme_void(base_size = 12)
}

seat_plot +
  facet_wrap(~ panel_label, ncol = 2) +
  guides(colour = "none") +
  labs(
    title = "Sitzverteilungen in ausgewählten Parlamenten",
    subtitle = "Jeder Punkt entspricht einem Sitz",
    caption = "Daten: PPEG 2025v1; Darstellung: ggparliament oder Fallback-Punktdiagramm"
  ) +
  theme(
    plot.title = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey30"),
    strip.text = element_text(face = "bold")
  )


### Stimmen und Sitze direkt vergleichen

Parlamentsdiagramme sind anschaulich. Für genaue Vergleiche sind Balkendiagramme oft besser. Hier sieht man direkt, welche Parteien mehr oder weniger Sitze erhalten als ihr Stimmenanteil erwarten ließe.


In [ ]:
seat_vote_long <- seat_examples |>
  select(panel_label, party_label, vote_share, seat_share) |>
  pivot_longer(
    cols = c(vote_share, seat_share),
    names_to = "measure",
    values_to = "share"
  ) |>
  mutate(
    measure = recode(
      measure,
      "vote_share" = "Stimmen",
      "seat_share" = "Sitze"
    )
  )

ggplot(seat_vote_long, aes(x = share, y = fct_reorder(party_label, share), fill = measure)) +
  geom_col(position = "dodge", width = 0.7) +
  facet_wrap(~ panel_label, scales = "free_y", ncol = 2) +
  scale_x_continuous(labels = scales::label_percent(scale = 1)) +
  scale_fill_manual(values = c("Stimmen" = "grey70", "Sitze" = "#2c7fb8")) +
  labs(
    title = "Stimmen- und Sitzanteile im Vergleich",
    x = "Anteil",
    y = NULL,
    fill = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(
    legend.position = "top",
    panel.grid.major.y = element_blank(),
    strip.text = element_text(face = "bold")
  )


## 7. ENP mit echten Wahldaten

Für echte Wahldaten berechnen wir zwei Varianten:

- `enp_votes`: effektive Parteienzahl auf Basis von Stimmenanteilen.
- `enp_seats`: effektive Parteienzahl auf Basis von Sitzanteilen.

Wenn `enp_seats` kleiner ist als `enp_votes`, hat das Wahlsystem die Parteienlandschaft im Parlament konzentriert.


In [ ]:
gallagher_from_data <- function(votes, seats) {
  keep <- !is.na(votes) & !is.na(seats)

  votes <- votes[keep]
  seats <- seats[keep]

  votes <- votes / sum(votes) * 100
  seats <- seats / sum(seats) * 100

  sqrt(0.5 * sum((votes - seats)^2))
}

election_indices <- ppeg_parl |>
  filter(!is.na(v_share_wgt), !is.na(s_share), !is.na(seats)) |>
  group_by(iso3c, country_name, election_date, election_year) |>
  summarise(
    parties_with_seats = sum(seats > 0, na.rm = TRUE),
    enp_votes = effective_number(v_share_wgt),
    enp_seats = effective_number(s_share),
    gallagher = gallagher_from_data(v_share_wgt, s_share),
    .groups = "drop"
  )

election_indices |>
  filter(iso3c == "DEU") |>
  arrange(election_date) |>
  tail(10)


In [ ]:
election_indices |>
  filter(iso3c == "DEU") |>
  ggplot(aes(x = election_date)) +
  geom_line(aes(y = enp_votes, colour = "Stimmen"), linewidth = 0.9) +
  geom_line(aes(y = enp_seats, colour = "Sitze"), linewidth = 0.9) +
  geom_point(aes(y = enp_votes, colour = "Stimmen"), size = 1.8) +
  geom_point(aes(y = enp_seats, colour = "Sitze"), size = 1.8) +
  scale_colour_manual(values = c("Stimmen" = "grey45", "Sitze" = "#2c7fb8")) +
  labs(
    title = "Effektive Parteienzahl in Deutschland",
    x = NULL,
    y = "Effektive Parteienzahl",
    colour = "Berechnet auf Basis von"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "top")


### Aufgabe 3: ENP in Ländern vergleichen

1. Ersetzen Sie in der vorherigen Zelle `"DEU"` durch `"NLD"`, `"GBR"`, `"USA"` oder `"ISR"`.
2. In welchem Land liegen `enp_votes` und `enp_seats` besonders weit auseinander?
3. Was sagt das über die Übersetzung von Stimmen in Sitze?
4. Passt das zu den Begriffen Zweiparteiensystem und Mehrparteiensystem aus der Vorlesung?


## 8. Disproportionalität: UK und Niederlande

Die folgende Aufgabe greift direkt eine typische Vergleichsfrage auf: Wie stark unterscheiden sich majoritäre und proportionale Wahlsysteme bei der Übersetzung von Stimmen in Sitze?


In [ ]:
latest_index <- function(country_code) {
  election_indices |>
    filter(iso3c == country_code) |>
    filter(election_date == max(election_date, na.rm = TRUE))
}

uk_nl_latest <- bind_rows(
  latest_index("GBR"),
  latest_index("NLD")
) |>
  mutate(
    country_label = case_when(
      iso3c == "GBR" ~ "Vereinigtes Königreich",
      iso3c == "NLD" ~ "Niederlande",
      TRUE ~ country_name
    )
  ) |>
  select(country_label, election_date, enp_votes, enp_seats, gallagher)

uk_nl_latest


In [ ]:
ggplot(uk_nl_latest, aes(x = country_label, y = gallagher)) +
  geom_col(fill = "#2c7fb8", width = 0.6) +
  geom_text(aes(label = round(gallagher, 1)), vjust = -0.35) +
  labs(
    title = "Disproportionalität bei der letzten Wahl",
    subtitle = "Vergleich UK und Niederlande",
    x = NULL,
    y = "Gallagher-Index"
  ) +
  theme_minimal(base_size = 12) +
  theme(panel.grid.major.x = element_blank())


### Aufgabe 4: Disproportionalität erklären

1. Welches Land hat den höheren Gallagher-Index: UK oder die Niederlande?
2. Schauen Sie mit `get_latest_election(ppeg_parl, "GBR")` und `get_latest_election(ppeg_parl, "NLD")` in die Parteidaten. Welche Partei profitiert besonders?
3. Welche Parteien verlieren besonders stark bei der Sitzverteilung?
4. Formulieren Sie in zwei Sätzen, wie das Wahlsystem den Parteienwettbewerb prägt.


In [ ]:
get_latest_election(ppeg_parl, "GBR") |>
  select(country_name, election_date, party_label, vote_share, seat_share, seats) |>
  arrange(desc(seats))


In [ ]:
get_latest_election(ppeg_parl, "NLD") |>
  select(country_name, election_date, party_label, vote_share, seat_share, seats) |>
  arrange(desc(seats))


## 9. Fragmentierung und Disproportionalität zusammen betrachten

Jetzt verbinden wir zwei Größen aus der Vorlesung:

- x-Achse: effektive Parteienzahl auf Basis der Stimmen.
- y-Achse: Disproportionalität der Stimmen-Sitz-Übersetzung.

Jeder Punkt ist eine Wahl.


In [ ]:
highlight_countries <- election_indices |>
  filter(iso3c %in% c("DEU", "GBR", "USA", "ISR", "NLD")) |>
  group_by(iso3c, country_name) |>
  filter(election_date == max(election_date, na.rm = TRUE)) |>
  ungroup()

ggplot(election_indices, aes(x = enp_votes, y = gallagher)) +
  geom_point(colour = "grey70", alpha = 0.35, size = 1.5) +
  geom_point(data = highlight_countries, colour = "#d95f02", size = 2.7) +
  ggrepel::geom_text_repel(
    data = highlight_countries,
    aes(label = iso3c),
    min.segment.length = 0,
    size = 3.5
  ) +
  labs(
    title = "Parteienfragmentierung und Disproportionalität",
    subtitle = "Jeder Punkt ist eine Wahl im PPEG-Datensatz",
    x = "Effektive Parteienzahl auf Basis der Stimmen",
    y = "Gallagher-Index"
  ) +
  theme_minimal(base_size = 12)


### Aufgabe 5: Muster im Streudiagramm

1. Wo liegen UK und USA im Vergleich zu Deutschland, Israel und den Niederlanden?
2. Gibt es viele Wahlen mit hoher Parteienfragmentierung und hoher Disproportionalität?
3. Warum ist es sinnvoll, Fragmentierung und Disproportionalität getrennt zu messen?


## 10. Parteipositionen in der Bundesrepublik

PPEG enthält keine direkten ideologischen Parteipositionen. Für diese Frage nutzen wir eine lokale ParlGov-CHES-Verknüpfung, sofern sie verfügbar ist.

Die Dimensionen folgen der Logik aus den Folien:

- x-Achse: wirtschaftliche Links-Rechts-Position.
- y-Achse: gesellschaftliche Links-Rechts-Position.

Die Werte werden auf eine 0-bis-10-Skala gebracht. Punktgröße entspricht dem Sitzanteil. Regierungsparteien werden anders markiert.


In [ ]:
load_position_data <- function() {
  position_path <- first_existing(c(
    "data/parlgov_ches.Rda",
    "D:/PolScience/Projekte/DATEN/parlgov2/parlgov/output/parlgov_ches.Rda"
  ))

  if (!is.na(position_path)) {
    message("Nutze lokale Positionsdaten: ", position_path)

    env <- new.env(parent = emptyenv())
    load(position_path, envir = env)

    return(env$parlgov_ches)
  }

  message("Keine lokalen Positionsdaten gefunden. Nutze ein kleines Lehrbeispiel.")

  tibble(
    country_name = "Germany",
    election_date = as.Date("2025-02-23"),
    party_name_short = c("Linke", "Grüne", "SPD", "FDP", "CDU/CSU", "AfD"),
    seat_share = c(10, 13, 19, 0, 33, 24),
    cabinet_party = c(FALSE, FALSE, FALSE, FALSE, TRUE, FALSE),
    lrecon_ppg = c(1.8, 3.2, 3.8, 7.8, 6.6, 5.8),
    galtan_ppg = c(3.5, 1.7, 3.7, 3.0, 6.0, 8.5)
  )
}

positions_raw <- load_position_data()


In [ ]:
rescale_position <- function(x) {
  if (max(x, na.rm = TRUE) <= 1.5) {
    return(x * 10)
  }

  x
}

germany_positions <- positions_raw |>
  filter(
    country_name == "Germany",
    !is.na(lrecon_ppg),
    !is.na(galtan_ppg),
    !is.na(seat_share),
    seat_share > 0
  ) |>
  group_by(country_name) |>
  filter(election_date == max(election_date, na.rm = TRUE)) |>
  ungroup() |>
  mutate(
    party_label = party_name_short,
    economic = rescale_position(lrecon_ppg),
    cultural = rescale_position(galtan_ppg),
    cabinet_party = case_when(
      cabinet_party %in% c(TRUE, 1) ~ "ja",
      TRUE ~ "nein"
    )
  ) |>
  distinct(party_label, .keep_all = TRUE)

germany_positions |>
  select(election_date, party_label, seat_share, economic, cultural, cabinet_party) |>
  arrange(desc(seat_share))


In [ ]:
ggplot(germany_positions, aes(x = economic, y = cultural)) +
  geom_vline(xintercept = 5, colour = "grey75") +
  geom_hline(yintercept = 5, colour = "grey75") +
  geom_point(
    aes(size = seat_share, shape = cabinet_party),
    colour = "#2c7fb8",
    alpha = 0.8
  ) +
  ggrepel::geom_text_repel(aes(label = party_label), min.segment.length = 0) +
  scale_x_continuous(limits = c(0, 10), breaks = 0:10) +
  scale_y_continuous(limits = c(0, 10), breaks = 0:10) +
  scale_size_continuous(range = c(3, 12), labels = scales::label_percent(scale = 1)) +
  scale_shape_manual(values = c("nein" = 16, "ja" = 17)) +
  labs(
    title = "Zweidimensionale Positionierung der Parteien in der Bundesrepublik",
    subtitle = paste("Letzte verfügbare Wahl in den lokalen Positionsdaten:", max(germany_positions$election_date)),
    x = "Wirtschaft: links/staatlich  -> rechts/marktliberal",
    y = "Gesellschaft: libertär/kosmopolitisch  -> autoritär/traditionalistisch",
    size = "Sitzanteil",
    shape = "Regierungspartei"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "top")


### Parteidistanzen berechnen

Ein Positionsraum erlaubt mehr als nur Visualisierung. Wir können auch berechnen, welche Parteien nah beieinander oder weit voneinander entfernt sind. Die folgende Zelle berechnet euklidische Distanzen im zweidimensionalen Raum.


In [ ]:
party_distances <- germany_positions |>
  select(party_label, economic, cultural) |>
  tidyr::crossing(
    germany_positions |>
      select(other_party = party_label, other_economic = economic, other_cultural = cultural)
  ) |>
  filter(party_label < other_party) |>
  mutate(
    distance = sqrt((economic - other_economic)^2 + (cultural - other_cultural)^2)
  ) |>
  arrange(desc(distance))

party_distances


### Aufgabe 6: Positionen und Cleavages

1. Welche Parteien liegen in Deutschland wirtschaftspolitisch nah beieinander, aber gesellschaftspolitisch weiter auseinander?
2. Welche zwei Parteien haben die größte Distanz im zweidimensionalen Raum?
3. Welche Konfliktlinie könnte die y-Achse abbilden: Klasse, Religion, Stadt/Land, Postmaterialismus, Migration/Integration?
4. Erfüllt diese Konfliktlinie die drei Cleavage-Kriterien nach Lipset/Rokkan: Sozialstruktur, kollektive Identität, organisatorischer Ausdruck?


## 11. Arbeitsblatt: Parteiensysteme in NRW

Das Arbeitsblatt vergleicht die Landtagswahlen in Nordrhein-Westfalen 1990 und 2022. Es fragt nach drei Größen:

- **ENEP**: effektive Anzahl elektoraler Parteien, also auf Basis der Stimmenanteile.
- **ENLP**: effektive Anzahl legislativer Parteien, also auf Basis der Sitzanteile.
- **Gallagher-Index**: Disproportionalität zwischen Stimmen und Sitzen.

Wir geben die Werte aus dem Arbeitsblatt zunächst als kleine Tabelle ein. Die Stimmenanteile sind gerundete Prozentwerte; deshalb normalisieren wir sie für die Berechnung intern wieder auf 100 Prozent.


In [ ]:
nrw_results <- tibble::tribble(
  ~year, ~party, ~votes, ~seats,
  2022, "CDU",       35.7, 76,
  2022, "SPD",       26.7, 56,
  2022, "Grüne",     18.2, 39,
  2022, "FDP",        5.9, 12,
  2022, "AfD",        5.4, 12,
  2022, "Linke",      2.1,  0,
  2022, "Sonstige",   6.1,  0,
  1990, "CDU",       36.7, 89,
  1990, "SPD",       50.0, 122,
  1990, "Grüne",      5.0, 12,
  1990, "FDP",        5.8, 14,
  1990, "Sonstige",   2.5,  0
)

nrw_results


In [ ]:
normalize_percent <- function(x) {
  x / sum(x, na.rm = TRUE) * 100
}

nrw_indices <- nrw_results |>
  group_by(year) |>
  mutate(
    vote_percent = normalize_percent(votes),
    seat_percent = normalize_percent(seats)
  ) |>
  summarise(
    ENEP = effective_number(vote_percent),
    ENLP = effective_number(seat_percent),
    Gallagher = gallagher_index(vote_percent, seat_percent),
    .groups = "drop"
  ) |>
  arrange(year)

nrw_indices


In [ ]:
nrw_long <- nrw_indices |>
  pivot_longer(
    cols = c(ENEP, ENLP, Gallagher),
    names_to = "index",
    values_to = "value"
  )

ggplot(nrw_long, aes(x = factor(year), y = value, fill = index)) +
  geom_col(position = "dodge", width = 0.7) +
  geom_text(
    aes(label = round(value, 1)),
    position = position_dodge(width = 0.7),
    vjust = -0.3,
    size = 3.5
  ) +
  scale_fill_manual(values = c("ENEP" = "#7f8c8d", "ENLP" = "#2c7fb8", "Gallagher" = "#d95f02")) +
  labs(
    title = "Parteiensystem NRW: 1990 und 2022",
    subtitle = "Effektive Parteienzahl und Disproportionalität aus den Arbeitsblattdaten",
    x = "Landtagswahl",
    y = "Wert",
    fill = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "top")


### Positionen der Parteien im zweidimensionalen Raum

Das Arbeitsblatt enthält Stimmen und Sitze, aber keine Positionsdaten. Für die Gruppenarbeit ergänzen wir deshalb einfache, didaktische Positionswerte für die im Arbeitsblatt vorkommenden Parteien.

Die x-Achse beschreibt die wirtschaftspolitische Dimension von links/staatlich bis rechts/marktliberal. Die y-Achse beschreibt die gesellschaftspolitische Dimension von libertär/kosmopolitisch bis autoritär/traditionalistisch. `Sonstige` wird nicht positioniert, weil dahinter verschiedene Parteien stehen.


In [ ]:
nrw_party_positions <- tibble::tribble(
  ~party,   ~economic, ~cultural,
  "Linke",      1.5,       4.5,
  "Grüne",      3.2,       1.8,
  "SPD",        3.8,       3.8,
  "FDP",        7.8,       3.2,
  "CDU",        6.4,       6.1,
  "AfD",        5.8,       8.6
)

nrw_positions_plot_data <- nrw_results |>
  filter(party != "Sonstige") |>
  left_join(nrw_party_positions, by = "party") |>
  mutate(
    in_parliament = seats > 0,
    year = factor(year)
  )

nrw_positions_plot_data


In [ ]:
ggplot(nrw_positions_plot_data, aes(x = economic, y = cultural)) +
  geom_vline(xintercept = 5, colour = "grey75") +
  geom_hline(yintercept = 5, colour = "grey75") +
  geom_point(
    aes(size = votes, alpha = in_parliament),
    colour = "#2c7fb8"
  ) +
  ggrepel::geom_text_repel(aes(label = party), min.segment.length = 0) +
  facet_wrap(~ year) +
  scale_x_continuous(limits = c(0, 10), breaks = 0:10) +
  scale_y_continuous(limits = c(0, 10), breaks = 0:10) +
  scale_size_continuous(range = c(3, 12), labels = scales::label_percent(scale = 1)) +
  scale_alpha_manual(values = c(`FALSE` = 0.35, `TRUE` = 0.9)) +
  labs(
    title = "Parteien des NRW-Arbeitsblatts im zweidimensionalen Raum",
    subtitle = "Punktgröße entspricht dem Stimmenanteil; blass = nicht im Landtag vertreten",
    x = "Wirtschaft: links/staatlich -> rechts/marktliberal",
    y = "Gesellschaft: libertär/kosmopolitisch -> autoritär/traditionalistisch",
    size = "Stimmenanteil",
    alpha = "Im Landtag"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "top")


### Aufgabe 7: Arbeitsblatt auswerten

1. Vergleichen Sie ENEP und ENLP für 1990 und 2022. Wann ist der Unterschied größer?
2. Warum ist die ENLP 1990 trotz vier Landtagsparteien relativ niedrig?
3. Welche Parteien sind 2022 elektoral sichtbar, aber parlamentarisch nicht oder kaum relevant?
4. Wie verändert sich der zweidimensionale Wettbewerbsraum zwischen 1990 und 2022?
5. Diskutieren Sie: Ist 2022 eher ein fragmentierteres Parteiensystem, ein stärker polarisiertes Parteiensystem oder beides?


## 12. Gruppenarbeit: Arbeitsaufträge

Teilen Sie sich in Gruppen auf. Jede Gruppe bearbeitet eine Aufgabe und bereitet eine kurze Interpretation vor. Wichtig ist nicht nur der berechnete Wert, sondern die politische Deutung.

### Gruppe A: Zweiparteiensysteme und Mehrparteiensysteme

Vergleichen Sie UK, USA, Deutschland und Israel. Nutzen Sie Sitzverteilungen, `enp_votes` und `enp_seats`. Passt die empirische Messung zu den Begriffen aus der Vorlesung?

### Gruppe B: Disproportionalität UK vs. Niederlande

Berechnen und erklären Sie die Disproportionalität bei den letzten Wahlen in UK und den Niederlanden. Welche Parteien gewinnen oder verlieren durch die Sitzübersetzung?

### Gruppe C: Bundesrepublik im Positionsraum

Interpretieren Sie die zweidimensionale Positionierung der Parteien in Deutschland. Welche Konfliktlinien strukturieren den Wettbewerb?

### Gruppe D: Experimentelle Parteiensysteme

Verändern Sie im ENP-Experiment Zahl und Größe der Parteien. Entwickeln Sie drei Parteiensysteme mit derselben Anzahl von Parteien, aber deutlich unterschiedlicher ENP.

### Gruppe E: Experimentelle Disproportionalität

Verändern Sie im Gallagher-Experiment Stimmen und Sitze. Entwickeln Sie ein Beispiel mit niedriger und eines mit hoher Disproportionalität. Erklären Sie, warum der Index reagiert.


In [ ]:
# Vorlage für eigene Länderanalysen
country_code <- "DEU"

latest_election <- get_latest_election(ppeg_parl, country_code)
latest_indices <- latest_index(country_code)

latest_indices


In [ ]:
latest_election |>
  select(country_name, election_date, party_label, vote_share, seat_share, seats) |>
  arrange(desc(seats))


## 13. Was man mitnehmen sollte

Parteien reduzieren Komplexität, organisieren Konflikte und verbinden Gesellschaft mit Regierung. Parteiensysteme kann man empirisch beschreiben, aber unterschiedliche Kennzahlen erfassen unterschiedliche Dinge:

- Die absolute Zahl der Parteien sagt wenig über deren politische Bedeutung.
- Die effektive Parteienzahl berücksichtigt die relative Größe der Parteien.
- Der Gallagher-Index misst die Verzerrung zwischen Stimmen und Sitzen.
- Positionsräume zeigen, dass Parteienwettbewerb oft mehrdimensional ist.
- Cleavages entstehen nicht automatisch aus sozialen Unterschieden, sondern wenn soziale Struktur, kollektive Identität, Organisation und Parteienrepräsentation dauerhaft zusammenkommen.
